# 2.7 Assignment Expressions and the Walrus Operator

## Overview

The assignment-expression operator (`:=`), introduced in Python 3.8, simultaneously **binds a simple name** and **produces the assigned value as an expression result**. Its correct use requires understanding lexical scoping, operator precedence, grammar restrictions, comprehension internals, evaluation order, short-circuiting, generator laziness, and how side effects interact with name binding.

The walrus operator is neither a replacement for ordinary assignment nor an in-place operator. Advanced usage must preserve explicit data flow and prevent confusing, hidden state changes.

### Learning Objectives

By the end of this notebook, you will be able to:

1. State the formal syntax and semantics of a named expression.
2. Distinguish `=`, `:=`, `==`, and augmented assignment.
3. Explain operator precedence and contexts requiring parentheses.
4. Prevent truthiness errors when a valid value is zero or an empty container.
5. Use captured expressions correctly in `if`, `while`, and short-circuit logic.
6. Apply assignment expressions in comprehensions without unintended scope effects.
7. Recognize prohibited comprehension positions, target collisions, and class-scope restrictions.
8. Explain generator-expression laziness and `any()` / `all()` witness capture.
9. Demonstrate key-before-value evaluation in dictionary comprehensions.
10. Analyze lambda, closure, `global`, and `nonlocal` interactions.
11. Verify single evaluation of expensive and side-effecting calculations.
12. Analyze `ast.NamedExpr` and compiler symbol tables.
13. Design readable, testable streaming data-processing code.
14. Build a deterministic, point-in-time quantitative market-data quality gate.
15. Verify correctness using numerical, temporal, and assignment-semantics invariants.

### Scope and Boundaries

This notebook focuses on **named expressions**. Ordinary assignment-target semantics and augmented in-place operations were developed in Sections 2.2–2.6. Advanced closure and scope mechanics will also be revisited later.

**Methodology:** Each language topic contains *Theoretical Foundations*, *Advanced Executable Example*, and a separate **Analytical Interpretation**. All examples use the standard library and include explicit assertions; the capstone includes automated tests, a reference report, advanced exercises, and an integrated challenge.

**Reproducibility:** Run cells top-to-bottom in a fresh kernel. Do not infer correctness from notebook cell execution counts alone.

## 2.7.1 Formal Semantics and the Boundary Between Statements and Expressions

### Theoretical Foundations

### Definition

A named expression has the form `NAME := expression`. The right-hand expression is evaluated once; the resulting object is **bound to the simple identifier** `NAME`, and the same object is returned as the value of the named expression.

| Construct | Syntax | Outcome |
|---|---|---|
| Ordinary assignment | `name = object` | Statement; no expression value |
| Named expression | `(name := object)` | Expression result **and** name binding |
| Equality comparison | `name == object` | Boolean or object-specific comparison result |
| Augmented assignment | `name += object` | Statement; may invoke in-place behavior |

Assignment-expression targets are **names**, not attributes, subscriptions, or unpacking patterns. Thus `(x := 3)` is valid while `(obj.x := 3)` and `(a[0] := 3)` are not.

A named expression **does not introduce a new lexical scope**. Its binding scope is determined by the containing syntactic context, subject to special comprehension rules.

**Important:** The value returned by `:=` is not automatically converted to `bool`. The enclosing expression or control-flow construct decides how to use it.

In [1]:
import ast

value = (captured := {"AAPL": 100, "MSFT": 50})
assert value is captured
assert isinstance(captured, dict)

# One expression, one value, two references to the same object.
assert (sample := 0) == 0
assert sample == 0

source = "(metric := quantity * price)"
tree = ast.parse(source, mode="eval")
assert isinstance(tree.body, ast.NamedExpr)
assert isinstance(tree.body.target, ast.Name)
assert isinstance(tree.body.target.ctx, ast.Store)
assert isinstance(tree.body.value, ast.BinOp)

print("Named-expression value:", value)
print("AST root:", type(tree.body).__name__)
print("AST target:", tree.body.target.id)

Named-expression value: {'AAPL': 100, 'MSFT': 50}
AST root: NamedExpr
AST target: metric


### **Analytical Interpretation**

**1. Semantic duality.** `:=` binds a name and yields the same object; it does not create a separate copy. **2. Target restriction.** The grammar permits a simple name target, not destructuring, attributes, or item targets. **3. AST structure.** `NamedExpr` has a `Name(Store)` target and a separate value subtree. **4. Runtime typing.** A captured zero remains a legitimate integer, not an automatically coerced Boolean. **5. Quant relevance.** The operator can preserve a computed intermediate result for reuse, provided the intermediate binding is visible and well-scoped.

## 2.7.2 Precedence, Parentheses, and Compile-Time Restrictions

### Theoretical Foundations

### Precedence

Named expressions have very low precedence, **below conditional expressions**. Parentheses often make the intended binding explicit.

- `(x := 3)`: valid standalone expression statement.
- `if (x := calculate()) > limit:`: comparison uses the captured value.
- `if x := calculate():`: valid and tests the captured value's truthiness.
- `value = (x := calculate())`: assignment statement whose RHS contains a named expression.
- `x := 3` as a bare statement: invalid without parentheses.

Parentheses are **required** in several syntactic contexts (including expression statements and as a subexpression in assignment statements, `assert`, `with`, `lambda`, conditional expressions, slices, keyword arguments, and comprehension filters). They are not universally required in `if` / `while` tests.

The outer target of `:=` cannot be an attribute, item, or tuple. A syntax failure should be tested via `compile()` rather than executing invalid source. Exact exception messages are version-dependent.

In [2]:
syntax_examples = {
    "parenthesized standalone": "(x := 3)",
    "if test": "if x := 3:\n    pass",
    "parenthesized comparison": "if (x := 3) > 2:\n    pass",
    "captured assignment RHS": "y = (x := 3)",
    "bare named expression": "x := 3",
    "attribute target": "(object.attribute := 1)",
    "item target": "(items[0] := 1)",
    "unpacking target": "((a, b) := (1, 2))",
    "assert without parentheses": "assert x := 3",
    "keyword without parentheses": "dict(a=x := 3)",
}
expected_valid = {
    "parenthesized standalone", "if test",
    "parenthesized comparison", "captured assignment RHS",
}
for title, source in syntax_examples.items():
    try:
        compile(source, "<named-expression-demo>", "exec")
        is_valid = True
    except SyntaxError:
        is_valid = False
    assert is_valid == (title in expected_valid), title
    print(f"{title:<33} {'VALID' if is_valid else 'SyntaxError'}")

# Comparison checks the capture, not the other way around.
if (observed := 0.035) > 0.03:
    risk_flag = "BREACH"
assert observed == 0.035 and risk_flag == "BREACH"

parenthesized standalone          VALID
if test                           VALID
parenthesized comparison          VALID
captured assignment RHS           VALID
bare named expression             SyntaxError
attribute target                  SyntaxError
item target                       SyntaxError
unpacking target                  SyntaxError
assert without parentheses        SyntaxError
keyword without parentheses       SyntaxError


### **Analytical Interpretation**

**1. Parentheses can be semantic protection.** `if (observed := f()) > limit` makes clear that `observed` receives the numeric result, not a comparison Boolean. **2. Compilation is not execution.** `compile()` validates language placement without running the snippets. **3. Target grammar.** Invalid `attribute`, `subscription`, and unpacking targets are rejected syntactically. **4. Version considerations.** Compile error text may vary; tests assert validity rather than message wording. **5. Financial clarity.** Distinguish a captured return or risk metric from its threshold decision.

## 2.7.3 Conditional Capture and the None-Versus-Falsy Distinction

### Theoretical Foundations

### Correctness before concision

`if value := get_value():` executes the block only for **truthy** values. That is appropriate only when zero, `False`, and empty collections should mean “do not proceed.” Missing market information is usually represented by `None`, which must be tested explicitly.

Use `if (value := read_metric()) is not None:` when **zero is valid**. Avoid `if value := ...:` when zero marks a legitimate observation, such as zero tracking error, zero volume, or zero return.

`None` must be compared using `is` / `is not`, never merely `==` when checking for absence. Comparisons with arbitrary user-defined objects may have domain-specific equality semantics.

In [3]:
from typing import Optional

observations: dict[str, Optional[float]] = {
    "AAPL": 0.0,
    "MSFT": 0.015,
    "JPM": None,
}

def read_return(ticker: str) -> Optional[float]:
    return observations.get(ticker)

accepted: dict[str, float] = {}
for ticker in ("AAPL", "MSFT", "JPM"):
    if (daily_return := read_return(ticker)) is not None:
        accepted[ticker] = daily_return

assert accepted == {"AAPL": 0.0, "MSFT": 0.015}
assert (zero_return := read_return("AAPL")) == 0.0
assert not bool(zero_return)
print("Valid observations (including zero):", accepted)

Valid observations (including zero): {'AAPL': 0.0, 'MSFT': 0.015}


### **Analytical Interpretation**

**1. Missing and zero are not equivalent.** A zero daily return is a valid observation. **2. Truthiness is a policy decision.** `if value := ...` would inadvertently discard zero. **3. One lookup.** The capture avoids repeating `read_return(ticker)`. **4. Explicit typing.** `Optional[float]` describes the possibility of absence; runtime checks still matter. **5. Quant implication.** Missing-value handling can shift backtest sample size and statistics even when the underlying formula is correct.

## 2.7.4 While Loops, Sentinels, and Streaming Consumption

### Theoretical Foundations

### Streaming semantics

A common pattern is `while (item := next(iterator, sentinel)) is not sentinel:`. An **identity** sentinel avoids interpreting valid falsy values as end-of-stream. A loop using `while (item := next(iterator, None)) is not None:` is suitable only if `None` is not a valid data event.

An expression is evaluated on each iteration and the target remains bound to the last value produced by the expression—including the terminal sentinel—after loop exit (unless enclosing scope lifetimes intervene).

For a `read()` API returning an empty bytes object on EOF, `while chunk := stream.read(8192):` can be appropriate. In market-data feeds, however, an empty dictionary or a numerical zero may be **valid input** and should not terminate consumption.

In [4]:
_SENTINEL = object()
stream = iter([0, {}, {"price": 100}, False, ""])
seen = []

while (record := next(stream, _SENTINEL)) is not _SENTINEL:
    seen.append(record)

assert seen == [0, {}, {"price": 100}, False, ""]
assert record is _SENTINEL

# Contrast: the truthiness variant incorrectly stops at zero.
stream2 = iter([0, 1, 2])
incorrectly_consumed = []
while item := next(stream2, None):
    incorrectly_consumed.append(item)
assert incorrectly_consumed == []
print("Identity-sentinel consumption:", seen)
print("Incorrect truthiness consumption:", incorrectly_consumed)

Identity-sentinel consumption: [0, {}, {'price': 100}, False, '']
Incorrect truthiness consumption: []


### **Analytical Interpretation**

**1. Iterative assignment.** The target is rebound once for every attempted fetch. **2. Sentinel identity.** Distinct sentinel objects preserve meaningful falsy events. **3. Termination binding.** At exit, the loop variable can hold the sentinel. **4. Stream completeness.** Terminating on `0` or `{}` can silently discard later events. **5. Production caution.** External streams also need cancellation, I/O errors, backpressure, and resource policies; a walrus loop alone does not supply them.

## 2.7.5 Boolean Short-Circuiting, Witnesses, and Side Effects

### Theoretical Foundations

### Short-circuit semantics

In `and` chains, a later expression is evaluated only if earlier operands are truthy; in `or` chains, evaluation stops when a truthy operand appears. Named-expression bindings in skipped branches **do not occur**. `and` / `or` normally return an operand object rather than necessarily returning `bool`.

An important implication: if a name is first assigned by a conditionally skipped named expression, code that later accesses that name may encounter an unbound or stale binding. Avoid relying on such bindings outside the branch that guarantees evaluation.

Parenthesized captures make guard order explicit: first check presence; then perform expensive calculations.

In [5]:
execution_events: list[str] = []

def load_optional() -> dict[str, float] | None:
    execution_events.append("load")
    return None

def expensive_risk(_: dict[str, float]) -> float:
    execution_events.append("compute")
    return 1.0

# Second branch is skipped because the first is None.
result = (
    (payload := load_optional()) is not None
    and (score := expensive_risk(payload)) > 0.5
)
assert result is False
assert execution_events == ["load"]

# Separate controlled example with an available payload.
execution_events.clear()

def get_payload() -> dict[str, float]:
    execution_events.append("load")
    return {"exposure": 0.7}

approved = (
    (payload := get_payload()) is not None
    and (risk_score := expensive_risk(payload)) <= 1.5
)
assert approved is True
assert execution_events == ["load", "compute"]
assert risk_score == 1.0
print("Validated guard trace:", execution_events)

Validated guard trace: ['load', 'compute']


### **Analytical Interpretation**

**1. Guard-first evaluation.** The expensive computation is skipped for missing input. **2. Conditional bindings.** A skipped named expression does not establish a new binding; in notebooks, a previous value might survive and become a misleading stale result. **3. Operand results.** A Boolean comparison returns `bool`, but plain `and` / `or` may return non-Boolean operands. **4. Financial integrity.** Costly risk metrics should run only after complete validation; do not hide essential state changes in conditional operands.

## 2.7.6 Comprehension Filters, Binding Scope, and Leakage

### Theoretical Foundations

### Comprehension-specific rule

A walrus target in a list, set, or dictionary comprehension (or generator expression) is bound in the **containing scope**, not the comprehension's iteration-variable scope. The iteration variable normally stays local to the comprehension's implementation-defined scope.

For example, `[z for x in items if (z := f(x)) is not None]` may leave `z` bound in the surrounding function after the comprehension executes. The binding records the **most recent evaluated value**, which is not necessarily the last included output.

An empty comprehension may perform **no assignment at all** to the walrus target. Never assume a new value exists following an empty input. For reproducible functions, initialize an intended outside binding first, and keep names local to each function call.

In [6]:
def capture_valid_metrics(records: list[float | None]):
    last_evaluated: float | None = None
    metrics = [
        observed
        for record in records
        if (observed := record) is not None
    ]
    last_evaluated = observed if records else None
    return metrics, last_evaluated

metrics, last_value = capture_valid_metrics([None, 0.0, 0.02, None])
assert metrics == [0.0, 0.02]
assert last_value is None  # Last *evaluated*, not last accepted.

empty_metrics, empty_last = capture_valid_metrics([])
assert empty_metrics == [] and empty_last is None

# Demonstrate enclosing-function binding distinctly from loop target.
def scope_demo():
    witness = "initial"
    numbers = [(witness := x * 2) for x in range(3)]
    return numbers, witness, "x" in locals()

numbers, witness, iter_var_leaked = scope_demo()
assert numbers == [0, 2, 4]
assert witness == 4
assert iter_var_leaked is False
print("Captured filter:", metrics, "last evaluated:", last_value)
print("Comprehension witness:", witness, "loop variable leaked:", iter_var_leaked)

Captured filter: [0.0, 0.02] last evaluated: None
Comprehension witness: 4 loop variable leaked: False


### **Analytical Interpretation**

**1. Scope is surprising.** The comprehension iteration name does not leak, while the named-expression target binds in the containing function. **2. Filter capture is precise.** A captured `None` from the final record overwrites the previous witness even though `None` is excluded. **3. Empty input.** A target may never be bound; explicit initialization avoids ambiguity. **4. Naming discipline.** Avoid using one walrus witness across unrelated comprehensions. **5. Quant relevance.** A stale statistic or unintended scope leak can silently distort downstream quality decisions.

## 2.7.7 Comprehension Restrictions, Iteration-Variable Collisions, and Class Bodies

### Theoretical Foundations

### Compiler restrictions

PEP 572 deliberately prohibits assignment expressions inside **comprehension iterable expressions** (the expressions following `in`). It also prohibits rebinding a comprehension's iteration variable through a named expression within the comprehension body or filters.

A walrus in a comprehension whose containing scope is a **class body** is forbidden when it would bind to that class's namespace; the restriction is rooted in the interaction between class execution namespaces and comprehension scoping.

These restrictions are compile-time behavior. Use `compile()` for diagnostics rather than executing invalid source. For portable examples, do not rely on the exact text of a `SyntaxError`.

In [7]:
invalid_sources = {
    "iterable expression": "[x for x in (captured := range(3))]",
    "nested iterable expression": "[x for x in range(3) for y in (items := [1, 2])]",
    "iteration-variable collision": "[(x := x + 1) for x in range(3)]",
    "class containing scope": "class C:\n    items = [(seen := x) for x in range(3)]",
}
for label, source in invalid_sources.items():
    try:
        compile(source, "<walrus-scope-test>", "exec")
    except SyntaxError:
        print(f"Rejected as expected: {label}")
    else:
        raise AssertionError(f"Expected a SyntaxError for {label}")

# The permitted variant binds in an ordinary enclosing function.
def legal_comprehension():
    latest = None
    output = [(latest := x * x) for x in range(4)]
    return output, latest

assert legal_comprehension() == ([0, 1, 4, 9], 9)

Rejected as expected: iterable expression
Rejected as expected: nested iterable expression
Rejected as expected: iteration-variable collision
Rejected as expected: class containing scope


### **Analytical Interpretation**

**1. Syntactic constraints matter.** A feature can be valid in ordinary expressions but invalid in specific comprehension positions. **2. Iteration target protection.** Binding an iteration variable via `:=` is not a supported way to modify the loop counter. **3. Class scope.** The class-body restriction avoids ambiguous binding semantics. **4. Detection without execution.** Compilation is sufficient to identify these prohibited forms. **5. Engineering guidance.** When the necessary code conflicts with grammar or readability, prefer an explicit `for` loop or helper function.

## 2.7.8 Generator Laziness and Any/All Witness Capture

### Theoretical Foundations

### Lazy evaluation and witnesses

Generator expressions defer most work until iteration. A `:=` embedded in the yielded expression or filter is therefore evaluated **only when the generator advances**. Initial creation of a generator does not necessarily bind the captured name.

`any()` stops on the first truthy result; `all()` stops on the first falsy result. A walrus can capture a witness or counterexample in the containing scope, but the final captured value depends on exactly how far the generator was consumed. If no element is evaluated, the capture may remain unbound (unless initialized).

Witness capture is useful for diagnostics but can obscure control flow; a dedicated explicit loop is often preferable for nontrivial applications.

In [8]:
def first_large_exposure(exposures: list[float], threshold: float):
    witness = None
    has_breach = any(
        (witness := (ticker, exposure))[1] > threshold
        for ticker, exposure in (
            (f"ASSET{i}", exposure)
            for i, exposure in enumerate(exposures)
        )
    )
    return has_breach, witness

assert first_large_exposure([0.10, 0.26, 0.40], 0.25) == (
    True, ("ASSET1", 0.26)
)
assert first_large_exposure([], 0.25) == (False, None)

# Generator is lazy: capture occurs only when next() is called.
def generator_capture():
    latest = None
    gen = ((latest := x * x) for x in [2, 3])
    before = latest
    first = next(gen)
    after = latest
    return before, first, after

assert generator_capture() == (None, 4, 4)
print("First breach:", first_large_exposure([0.10, 0.26, 0.40], 0.25))

First breach: (True, ('ASSET1', 0.26))


### **Analytical Interpretation**

**1. `any()` is lazy.** It stops after the first breach, so subsequent exposures are not visited. **2. Captured witness.** The bound tuple refers to the first breaching asset, not the maximum exposure. **3. Empty input.** Initialized `witness = None` remains unchanged. **4. Generator creation versus execution.** The capture in the generator does not happen until iteration advances. **5. Quant limitation.** An early-exit breach detector is not equivalent to a full portfolio risk report; choose an explicit exhaustive pass when all violations are required.

## 2.7.9 Dictionary Comprehension Key-Before-Value Evaluation

### Theoretical Foundations

### Evaluation order after PEP 572

For dictionary comprehensions, **the key expression is evaluated before the value expression** (defined in Python 3.8 alongside PEP 572). This enables a name captured while calculating the key to be reused while calculating the associated value.

Avoid relying on source-order intuition for complex expressions: create small deterministic traces when evaluating the order of side effects. Be careful about duplicate keys, which can replace previous values in the resulting dictionary; capture names may still be rebound on every iteration.

In [9]:
from decimal import Decimal

evaluation_trace: list[str] = []

def normalized_ticker(raw: str) -> str:
    evaluation_trace.append(f"key:{raw}")
    return raw.strip().upper()

def value_for(ticker: str, amount: str) -> Decimal:
    evaluation_trace.append(f"value:{ticker}")
    return Decimal(amount)

rows = [(" aapl ", "100"), (" msft ", "200")]
notional_by_ticker = {
    (ticker := normalized_ticker(raw)): value_for(ticker, amount)
    for raw, amount in rows
}
assert notional_by_ticker == {"AAPL": Decimal("100"), "MSFT": Decimal("200")}
assert evaluation_trace == [
    "key: aapl ", "value:AAPL", "key: msft ", "value:MSFT"
]
print("Dictionary:", notional_by_ticker)
print("Key/value evaluation trace:", evaluation_trace)

Dictionary: {'AAPL': Decimal('100'), 'MSFT': Decimal('200')}
Key/value evaluation trace: ['key: aapl ', 'value:AAPL', 'key: msft ', 'value:MSFT']


### **Analytical Interpretation**

**1. Key first.** Every dictionary-comprehension entry computes the key before the value. **2. Name reuse.** Capturing the normalized ticker avoids duplicated normalization. **3. Financial semantics.** Keys may be normalized identifiers used when calculating associated values. **4. Duplicate-key policy.** A dictionary silently overwrites earlier values for equal keys; a ledger should explicitly define whether to aggregate, reject, or overwrite. **5. Clarity.** Simple comprehensions are appropriate, but complex financial reconciliation should use named helper functions and explicit validation.

## 2.7.10 Lambda, Closures, Global/Nonlocal Scope and Binding

### Theoretical Foundations

### Scope rules

A lambda defines a new function scope; a named expression in its body binds in that lambda's scope. In ordinary functions, `:=` targets follow the applicable local, global, or `nonlocal` binding rules. A named expression itself does not introduce another scope.

To test scope behavior, return observable results rather than relying on incidental memory addresses. Captures in closures can affect future calls; avoid hidden mutable state where reproducibility matters.

In [10]:
scale = lambda x: (normalized := x / 100) * 10  # lambda-local normalized
assert scale(25) == 2.5
assert "normalized" not in globals()

def make_metric_counter():
    count = 0
    def record(value: float) -> tuple[float, int]:
        nonlocal count
        if (accepted := value) >= 0:
            count = count + 1
        return accepted, count
    return record

record_metric = make_metric_counter()
assert record_metric(0.0) == (0.0, 1)
assert record_metric(-1.0) == (-1.0, 1)
assert record_metric(3.0) == (3.0, 2)

def explicitly_local():
    captured = "prior"
    out = [captured := x for x in [4, 5]]
    return out, captured

assert explicitly_local() == ([4, 5], 5)
print("Closure counter after three records:", record_metric(1.0))

Closure counter after three records: (1.0, 3)


### **Analytical Interpretation**

**1. Lambda scope.** `normalized` is local to the lambda's execution, not a module-level binding. **2. `nonlocal` behavior.** The explicit declaration permits updates to the enclosing `count`. **3. Capture versus control.** `accepted` is a function-local name because it is assigned by the walrus in an ordinary function. **4. Comprehension contrast.** The capture in a comprehension can update the containing function binding. **5. Quant recommendation.** Stateful closures are useful for small instrumentation tasks, but persistent model state is generally easier to audit when represented explicitly.

## 2.7.11 Single Evaluation, Memoization, and Side-Effect Boundaries

### Theoretical Foundations

### The main legitimate use case

The walrus is useful when the same expensive calculation would otherwise appear multiple times in an expression. It can enforce **one evaluation per control-flow pass** without constructing additional temporary statements.

A named expression is not automatically memoization across iterations or calls. It captures **one evaluated result** at a specific point. Genuine memoization needs a cache and a cache invalidation policy.

It is a mistake to assume `:=` improves performance in all code: often ordinary assignment is equally efficient and more readable. Never write a named expression solely to make a one-line expression longer and harder to debug.

In [11]:
from decimal import Decimal

calls = {"spread": 0}

def spread_bps(bid: Decimal, ask: Decimal) -> Decimal:
    calls["spread"] += 1
    mid = (bid + ask) / 2
    return (ask - bid) / mid * Decimal("10000")

quotes = [
    (Decimal("99.95"), Decimal("100.05")),
    (Decimal("100.00"), Decimal("100.00")),
    (Decimal("99.00"), Decimal("101.00")),
]

max_spread_bps = Decimal("20")
spreads = [
    spread
    for bid, ask in quotes
    if (spread := spread_bps(bid, ask)) <= max_spread_bps
]

assert calls["spread"] == 3
assert len(spreads) == 2
assert spreads[1] == Decimal("0")
print("Accepted spreads:", spreads)
print("Spread calculations:", calls["spread"])

Accepted spreads: [Decimal('10.000'), Decimal('0')]
Spread calculations: 3


### **Analytical Interpretation**

**1. One call per quote.** `spread_bps` executes once for each input row. **2. Reuse in the filter.** The resulting value is used both for the filter and output. **3. Zero spread.** A locked quote has spread zero; whether such quotes should be allowed is a distinct market-data policy. **4. No automatic caching.** The name is rebound on subsequent iterations, not cached across them. **5. Research accuracy.** Transparent numerical calculations matter more than avoiding a harmless temporary assignment.

## 2.7.12 AST and Symbol-Table Auditing of Named Expressions

### Theoretical Foundations

### Static representation

Python represents `:=` using `ast.NamedExpr`, whose target is an `ast.Name` in `Store` context. The `ast` tree can identify named expressions, name collisions, and where captures appear in comprehensions. `symtable` exposes compiler classifications (local, global, free, and more) but **not** runtime values.

A static analyzer should treat a named expression as an observable binding operation. It should also record whether the capture appears in a generator, comprehension, or conditional where it may never execute.

Code-object / bytecode internals can change between Python versions; prefer AST when expressing source-level semantics. Do not execute unknown code merely to inspect binding usage.

In [12]:
import ast
import symtable

sample_source = """
def detect(values):
    previous = None
    return [captured for item in values if (captured := item) is not None]
"""

sample_tree = ast.parse(sample_source)
named_nodes = [
    node for node in ast.walk(sample_tree)
    if isinstance(node, ast.NamedExpr)
]
assert len(named_nodes) == 1
assert named_nodes[0].target.id == "captured"
assert isinstance(named_nodes[0].target.ctx, ast.Store)

symbols = symtable.symtable(sample_source, "<audit>", "exec")
fn_table = next(s for s in symbols.get_children() if s.get_name() == "detect")
captured_symbol = fn_table.lookup("captured")
assert captured_symbol.is_local()
assert captured_symbol.is_assigned()

print("Captured target:", named_nodes[0].target.id)
print("Function scope/local:", captured_symbol.is_local())
print("AST node:", ast.dump(named_nodes[0], include_attributes=False))

Captured target: captured
Function scope/local: True
AST node: NamedExpr(target=Name(id='captured', ctx=Store()), value=Name(id='item', ctx=Load()))


### **Analytical Interpretation**

**1. Binding visibility.** `NamedExpr` is a distinct node type, not ordinary `Assign`. **2. Scope classification.** The comprehension's capture is classified as local in the enclosing function. **3. Static-analysis limit.** AST structure does not establish that `captured` is initialized at runtime; empty iterables may skip the binding. **4. Security.** Static inspection is not equivalent to safe execution or sandboxing. **5. Quant application.** Source audits can identify hidden variable assignments in feature-engineering and risk-calculation expressions.

## 2.7.13 Readability, Errors, and Design Trade-offs

### Theoretical Foundations

### When not to use `:=`

Avoid using a named expression if:

1. A straightforward assignment statement is clearer.
2. Its name will persist into a broader scope unintentionally.
3. A condition short-circuits and later code assumes the target is initialized.
4. A comprehension obscures the meaning or evaluation order.
5. It may trigger a side effect while filtering data.
6. The resulting expression depends on a subtle precedence rule.
7. Failure paths would be easier to test with explicit statements.

`:=` is particularly useful for: a single expensive subexpression, a `while` read loop with a clear sentinel, or a simple `if (match := ...)` guard. It should be justified by **clarity and correct evaluation**, not novelty.

A subtle error: `(x := fallible())` does not guarantee a new binding when `fallible()` raises; earlier bindings may remain in the same namespace. Initialize or use function-local state when stale results are dangerous.

In [13]:
# The previous capture survives an exception if evaluation fails.
def demonstrate_stale_capture():
    captured = "previous-run"
    def failed_metric():
        raise ValueError("upstream observation invalid")
    try:
        (captured := failed_metric())
    except ValueError:
        pass
    return captured

assert demonstrate_stale_capture() == "previous-run"

# Prefer explicit statements for complex financial validation.
def clear_check(observation: dict[str, float]) -> tuple[bool, float | None]:
    raw = observation.get("score")
    if raw is None:
        return False, None
    if raw < 0 or raw > 1:
        return False, raw
    return True, raw

assert clear_check({"score": 0.0}) == (True, 0.0)
assert clear_check({"score": -0.1}) == (False, -0.1)
print("Previous capture after failed evaluation:", demonstrate_stale_capture())

Previous capture after failed evaluation: previous-run


### **Analytical Interpretation**

**1. Exceptions prevent rebinding.** If the RHS raises, the assignment-expression target is not newly assigned. **2. Stale state.** A previous name binding may survive and mislead later code. **3. Explicit validation wins for branching complexity.** A short named expression is not automatically preferable to a clear function. **4. Quant reliability.** Avoid an analyst's notebook depending on a variable that happened to be set during a previous cell execution. **5. Review standard.** Use `:=` only when the value is genuinely reused and the scope and error behavior are obvious.

## 2.7.14 Practical Application — Point-in-Time Market Data Quality Gate

### Problem formulation

A quantitative trading system receives a stream of market quotes. Each event contains a unique identifier, asset symbol, per-symbol sequence number, bid price, and ask price. A robust **quality gate** should reject invalid events before the quote becomes eligible for feature calculation or strategy evaluation.

The central walrus applications are: **one-pass field capture**, **sentinel-based streaming**, **guarded optional-value handling**, and **single-evaluation numerical thresholds**. Here, correctness also requires an explicit and deterministic event-processing policy.

### Model and research constraints

- Reject missing fields, Boolean sequence numbers, nonpositive prices, and nonfinite values.
- Reject crossed books (`bid > ask`) and duplicate IDs.
- Require strictly increasing **per-symbol sequence numbers** relative to the last *accepted* quote.
- Reject relative bid–ask spread above a fixed limit.
- Reject a mid-price jump above a fixed limit relative to the last accepted mid.
- Allow zero spread **as a deliberate synthetic-data policy**; production market policy may differ.
- A rejected quote **must not poison** the reference state for subsequent comparisons.
- Every input event produces exactly one decision, including malformed or empty events.
- Process data in deterministic supplied-stream order; no concurrency or reordering is modeled.

### Definitions

For bid \(b > 0\) and ask \(a > 0\), if \(b \leq a\):

$$
\operatorname{mid} = \frac{a+b}{2}
$$

$$
\operatorname{spread}_{\mathrm{bps}} = 10{,}000 \frac{a-b}{\operatorname{mid}}
$$

For a previously **accepted** mid \(m_0\):

$$
\operatorname{jump}_{\mathrm{bps}} = 10{,}000 \frac{|m_1-m_0|}{m_0}
$$

These metrics assess **synthetic quote quality**, not trading profitability.

`Decimal` is used to avoid introducing unintended binary floating-point conversions inside the metrics; the `Decimal` arithmetic context and precision remain relevant.


### Pipeline architecture

```text
Raw quote iterator
    |
    v
Identity-sentinel read loop
    |
    v
Walrus-based validation / Decimal normalization
    |
    v
Duplicate and temporal checks
    |
    v
Spread and mid-jump computations
    |
    +----> REJECT (keep previous accepted reference)
    |
    v
ACCEPT (update last-accepted reference)
    |
    v
Immutable decisions + deterministic report
```

**Deliberate scope restriction:** This is a single-process analytical quality gate, not a market-data feed handler with exchange timestamps, latency models, message recovery, or consolidated-quote semantics.

In [14]:
from collections.abc import Iterable, Mapping
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from typing import Literal

BPS = Decimal("10000")
DecisionStatus = Literal["ACCEPT", "REJECT"]


@dataclass(frozen=True, slots=True)
class QuoteGatePolicy:
    max_spread_bps: Decimal
    max_jump_bps: Decimal

    def __post_init__(self) -> None:
        for name in ("max_spread_bps", "max_jump_bps"):
            value = getattr(self, name)
            if not isinstance(value, Decimal) or not value.is_finite() or value < 0:
                raise ValueError(f"{name} must be a finite nonnegative Decimal")


@dataclass(frozen=True, slots=True)
class MarketQuote:
    event_id: str
    ticker: str
    sequence: int
    bid: Decimal
    ask: Decimal

    @property
    def mid(self) -> Decimal:
        return (self.bid + self.ask) / Decimal("2")


@dataclass(frozen=True, slots=True)
class QuoteDecision:
    event_id: str
    ticker: str | None
    status: DecisionStatus
    reason: str
    spread_bps: Decimal | None
    jump_bps: Decimal | None


@dataclass(frozen=True, slots=True)
class StreamReport:
    decisions: tuple[QuoteDecision, ...]
    accepted_quotes: tuple[MarketQuote, ...]
    last_accepted: tuple[tuple[str, MarketQuote], ...]

    @property
    def accepted_count(self) -> int:
        return len(self.accepted_quotes)

    @property
    def rejected_count(self) -> int:
        return len(self.decisions) - self.accepted_count


def parse_positive_decimal(value: object) -> Decimal | None:
    """Normalize int/float/str/Decimal values; reject bool and invalid prices."""
    if type(value) is bool or not isinstance(value, (int, float, str, Decimal)):
        return None
    try:
        normalized = Decimal(str(value).strip())
    except (InvalidOperation, ValueError):
        return None
    if not normalized.is_finite() or normalized <= 0:
        return None
    return normalized


def parse_market_quote(raw: object) -> MarketQuote | None:
    """Validate and normalize one raw quote without mutating source data."""
    if not isinstance(raw, Mapping):
        return None
    if not isinstance((event_id := raw.get("event_id")), str) or not event_id.strip():
        return None
    if not isinstance((ticker := raw.get("ticker")), str) or not ticker.strip():
        return None
    if (
        type(sequence := raw.get("sequence")) is not int
        or sequence < 0
    ):
        return None
    if (bid := parse_positive_decimal(raw.get("bid"))) is None:
        return None
    if (ask := parse_positive_decimal(raw.get("ask"))) is None:
        return None
    return MarketQuote(
        event_id=event_id.strip(),
        ticker=ticker.strip().upper(),
        sequence=sequence,
        bid=bid,
        ask=ask,
    )

POLICY = QuoteGatePolicy(
    max_spread_bps=Decimal("20"),
    max_jump_bps=Decimal("150"),
)
assert parse_positive_decimal(0) is None
assert parse_positive_decimal("0.00") is None
assert parse_positive_decimal("100.25") == Decimal("100.25")
assert parse_market_quote({}) is None
print("Market-quote models, policy, and walrus-based parser initialized.")

Market-quote models, policy, and walrus-based parser initialized.


### **Analytical Interpretation**

**1. Data boundary.** `parse_market_quote` is the sole normalization boundary for raw event dictionaries; invalid data yields `None` rather than a partially formed quote. **2. Walrus captures.** `event_id`, `ticker`, `sequence`, `bid`, and `ask` are evaluated once and can be reused after their guards. **3. Boolean rejection.** `type(sequence) is int` deliberately rejects `True` despite `bool` subclassing `int`. **4. Distinguish empty from missing.** The parser can reject an empty mapping, while the subsequent stream loop must still **count** it as an event. **5. Decimal policy.** Conversion from floats through `str()` reflects their displayed decimal representation; for monetary source systems, decimal strings are preferred. **6. Immutability.** Frozen records prevent ordinary attribute reassignment but are not a security boundary against arbitrary reflective mutation.

In [15]:
from collections import Counter


def assess_quote_stream(
    records: Iterable[object],
    policy: QuoteGatePolicy,
) -> StreamReport:
    """
    Evaluate an event stream with deterministic, point-in-time quality rules.

    Rejected quotes do not change the last-accepted reference.
    Every supplied event yields exactly one decision.
    """
    seen_valid_ids: set[str] = set()
    last_by_ticker: dict[str, MarketQuote] = {}
    accepted: list[MarketQuote] = []
    decisions: list[QuoteDecision] = []

    iterator = iter(records)
    sentinel = object()
    ordinal = 0

    while (raw := next(iterator, sentinel)) is not sentinel:
        ordinal += 1
        if (quote := parse_market_quote(raw)) is None:
            ident = raw.get("event_id") if isinstance(raw, Mapping) else None
            ticker = raw.get("ticker") if isinstance(raw, Mapping) else None
            decisions.append(
                QuoteDecision(
                    event_id=ident if isinstance(ident, str) and ident else f"MALFORMED#{ordinal}",
                    ticker=ticker if isinstance(ticker, str) else None,
                    status="REJECT",
                    reason="INVALID_QUOTE",
                    spread_bps=None,
                    jump_bps=None,
                )
            )
            continue

        def reject(reason: str, spread: Decimal | None = None, jump: Decimal | None = None):
            decisions.append(
                QuoteDecision(
                    quote.event_id, quote.ticker,
                    "REJECT", reason, spread, jump,
                )
            )

        if quote.event_id in seen_valid_ids:
            reject("DUPLICATE_ID")
            continue
        seen_valid_ids.add(quote.event_id)

        if quote.bid > quote.ask:
            reject("CROSSED_BOOK")
            continue

        previous = last_by_ticker.get(quote.ticker)
        if previous is not None and quote.sequence <= previous.sequence:
            reject("STALE_SEQUENCE")
            continue

        spread = (quote.ask - quote.bid) / quote.mid * BPS
        if (observed_spread := spread) > policy.max_spread_bps:
            reject("WIDE_SPREAD", observed_spread)
            continue

        # Jump is undefined for the first accepted quote of an asset.
        jump: Decimal | None = None
        if previous is not None:
            if (
                jump := abs(quote.mid - previous.mid) / previous.mid * BPS
            ) > policy.max_jump_bps:
                reject("MID_JUMP", observed_spread, jump)
                continue

        accepted.append(quote)
        last_by_ticker[quote.ticker] = quote
        decisions.append(
            QuoteDecision(
                quote.event_id,
                quote.ticker,
                "ACCEPT",
                "OK",
                observed_spread,
                jump,
            )
        )

    return StreamReport(
        decisions=tuple(decisions),
        accepted_quotes=tuple(accepted),
        last_accepted=tuple(sorted(last_by_ticker.items())),
    )


print("Deterministic quote stream gate initialized.")

Deterministic quote stream gate initialized.


### **Analytical Interpretation**

**1. Sentinel loop.** `while (raw := next(iterator, sentinel)) is not sentinel` processes every input event, including empty mappings, without confusing an empty event with end-of-stream. **2. Parser capture.** `(quote := parse_market_quote(raw)) is None` performs normalization exactly once per event. **3. Threshold capture.** The spread and jump values are preserved for rejection reports and are not recomputed. **4. Temporal integrity.** Comparisons refer to the **last accepted** quote per ticker. A rejected outlier never becomes the reference for the next event. **5. Scope and side effects.** `reject()` is a local helper with an explicit report-append effect; it is intentionally not hidden inside a Boolean expression. **6. Duplicate policy.** Every well-formed event ID is considered observed after syntax validation, even if its quote later fails a quality gate; malformed IDs are not reserved. **7. No transaction guarantee.** The function holds temporary local state and returns one immutable report, but does not provide distributed ordering or crash recovery.

In [16]:
# Each quote is a synthetic event, not a real exchange observation.
EXAMPLE_EVENTS = [
    {"event_id": "Q1", "ticker": "AAPL", "sequence": 100, "bid": "99.95", "ask": "100.05"},
    {"event_id": "Q2", "ticker": "AAPL", "sequence": 101, "bid": "100.00", "ask": "100.10"},
    {"event_id": "Q3", "ticker": "MSFT", "sequence": 100, "bid": "200.00", "ask": "200.00"},
    {"event_id": "Q4", "ticker": "AAPL", "sequence": 100, "bid": "100.00", "ask": "100.10"},
    {"event_id": "Q5", "ticker": "MSFT", "sequence": 102, "bid": "199.00", "ask": "201.00"},
    {"event_id": "Q6", "ticker": "MSFT", "sequence": 103, "bid": "208.00", "ask": "208.20"},
    {"event_id": "Q7", "ticker": "MSFT", "sequence": 104, "bid": "202.00", "ask": "202.10"},
    {},  # Deliberately malformed, but must not end the stream.
    {"event_id": "Q9", "ticker": "AAPL", "sequence": 105, "bid": "101.00", "ask": "100.00"},
    {"event_id": "Q10", "ticker": "AAPL", "sequence": 106, "bid": "100.05", "ask": "100.05"},
    {"event_id": "Q11", "ticker": "MSFT", "sequence": 99, "bid": "201.00", "ask": "201.10"},
    {"event_id": "Q12", "ticker": "AAPL", "sequence": 107, "bid": "100.00"},
    {"event_id": "Q13", "ticker": "AAPL", "sequence": 108, "bid": "100.10", "ask": "100.20"},
    {"event_id": "Q2", "ticker": "AAPL", "sequence": 109, "bid": "100.10", "ask": "100.20"},
]

stream_report = assess_quote_stream(iter(EXAMPLE_EVENTS), POLICY)
reasons = Counter(d.reason for d in stream_report.decisions)

assert len(stream_report.decisions) == 14
assert stream_report.accepted_count == 6
assert stream_report.rejected_count == 8
assert reasons == {
    "OK": 6,
    "INVALID_QUOTE": 2,
    "STALE_SEQUENCE": 2,
    "CROSSED_BOOK": 1,
    "WIDE_SPREAD": 1,
    "MID_JUMP": 1,
    "DUPLICATE_ID": 1,
}

print("=" * 91)
print("POINT-IN-TIME QUOTE QUALITY GATE — SYNTHETIC RESEARCH EXAMPLE")
print("=" * 91)
print(f"{'Event':<16}{'Ticker':<9}{'Status':<10}{'Reason':<20}{'Spread (bps)':>18}{'Jump (bps)':>18}")
print("-" * 91)
for d in stream_report.decisions:
    spread = "N/A" if d.spread_bps is None else f"{d.spread_bps:.3f}"
    jump = "N/A" if d.jump_bps is None else f"{d.jump_bps:.3f}"
    print(f"{d.event_id:<16}{(d.ticker or 'N/A'):<9}{d.status:<10}{d.reason:<20}{spread:>18}{jump:>18}")

print("\nSummary:")
print(f"  Events processed: {len(stream_report.decisions)}")
print(f"  Accepted: {stream_report.accepted_count}")
print(f"  Rejected: {stream_report.rejected_count}")
print("  Final accepted mids:", {ticker: str(q.mid) for ticker, q in stream_report.last_accepted})

POINT-IN-TIME QUOTE QUALITY GATE — SYNTHETIC RESEARCH EXAMPLE
Event           Ticker   Status    Reason                    Spread (bps)        Jump (bps)
-------------------------------------------------------------------------------------------
Q1              AAPL     ACCEPT    OK                              10.000               N/A
Q2              AAPL     ACCEPT    OK                               9.995             5.000
Q3              MSFT     ACCEPT    OK                               0.000               N/A
Q4              AAPL     REJECT    STALE_SEQUENCE                     N/A               N/A
Q5              MSFT     REJECT    WIDE_SPREAD                    100.000               N/A
Q6              MSFT     REJECT    MID_JUMP                         9.611           405.000
Q7              MSFT     ACCEPT    OK                               4.949           102.500
MALFORMED#8     N/A      REJECT    INVALID_QUOTE                      N/A               N/A
Q9              AA

### **Expected Results**

| Metric | Expected value |
|---|---:|
| Input events | **14** |
| Accepted | **6** |
| Rejected | **8** |
| Invalid quotes | 2 |
| Stale sequence | 2 |
| Crossed book | 1 |
| Spread too wide | 1 |
| Mid-price jump too large | 1 |
| Duplicate ID | 1 |
| Final accepted AAPL sequence | 108 |
| Final accepted AAPL mid | **100.15 USD** |
| Final accepted MSFT sequence | 104 |
| Final accepted MSFT mid | **202.05 USD** |
| Accepted MSFT quote Q3 spread | **0 bps** |
| Accepted AAPL quote Q10 spread | **0 bps** |

The model intentionally permits **zero spread** as a test policy; it is not a recommendation to accept every locked quote in production. Accepted status means **passes this synthetic gate**, not market validity, execution viability, or predictive quality.

### **Analytical Interpretation**

**1. Complete event coverage.** The empty dictionary is rejected as a malformed quote, not confused with the stream's termination sentinel. **2. Valid zero values.** The locked-book examples demonstrate a zero computed spread that is not treated as missing. **3. Relative reference integrity.** Q5 and Q6 do not poison MSFT's accepted reference, allowing Q7 to be evaluated against Q3. **4. Deterministic order.** Re-running the same input sequence produces identical reports. **5. Financial interpretation.** Spread and jump metrics in basis points are quality heuristics rather than portfolio returns. **6. Scope limitation.** Actual exchange feeds need event-time provenance, venue rules, clock synchronization, sequence recovery, depth data, and realistic operational constraints.

## 2.7.15 Automated Testing, Property Checks, and Invariants

### Test design

The test suite is intentionally divided into **language-semantic** tests and **domain-specific** tests. These examine different failure modes.

**Language semantics**

1. The named-expression value is the bound object.
2. Zero is distinguishable from `None`.
3. Sentinel loops preserve falsy events.
4. Comprehension captures affect the containing function.
5. Generator captures are lazy.
6. `any()` short-circuits at a witness.
7. Invalid grammar is rejected.
8. AST has a `NamedExpr(Name(Store), value)` structure.
9. Failed RHS evaluation does not produce a fresh binding.
10. Dictionary-comprehension keys precede their values.

**Market-data invariants**

1. `accepted + rejected == number_of_input_events`.
2. Every accepted quote has a unique event ID.
3. Accepted sequence numbers increase strictly per ticker.
4. Accepted quotes have `0 <= spread_bps <= policy.max_spread_bps`.
5. Accepted noninitial quotes have `0 <= jump_bps <= policy.max_jump_bps`.
6. Rejected events never replace the last-accepted reference.
7. Parser and evaluator do not mutate raw input dictionaries.
8. A deterministic input sequence produces an identical report.
9. Invalid price types, missing fields, nonfinite values, and negative policy limits are rejected.
10. Boundary conditions at the exact threshold must follow the specified **strict greater-than** breach rule.

**Statistical limitation:** The test data are synthetic. Passing these tests does not establish market-data accuracy, trading profitability, or absence of every possible implementation error.

In [17]:
import unittest


def quote_record(event_id, ticker="AAPL", sequence=1, bid="100", ask="100"):
    return {
        "event_id": event_id, "ticker": ticker, "sequence": sequence,
        "bid": bid, "ask": ask,
    }


class WalrusSemanticsTests(unittest.TestCase):
    def test_returns_bound_object(self):
        original = []
        self.assertIs((captured := original), original)
        self.assertIs(captured, original)

    def test_zero_capture(self):
        self.assertEqual((z := 0), 0)
        self.assertIsNotNone(z)

    def test_zero_comparison_guard(self):
        if (return_value := 0.0) is not None:
            accepted = True
        else:
            accepted = False
        self.assertTrue(accepted)

    def test_identity_sentinel_keeps_zero(self):
        stop = object()
        it = iter([0, None, {}, False])
        seen = []
        while (value := next(it, stop)) is not stop:
            seen.append(value)
        self.assertEqual(seen, [0, None, {}, False])

    def test_standalone_expression_requires_parentheses(self):
        with self.assertRaises(SyntaxError):
            compile("x := 1", "<test>", "exec")

    def test_parenthesized_expression(self):
        compile("(x := 1)", "<test>", "exec")

    def test_attribute_target_invalid(self):
        with self.assertRaises(SyntaxError):
            compile("(obj.x := 3)", "<test>", "exec")

    def test_subscription_target_invalid(self):
        with self.assertRaises(SyntaxError):
            compile("(data[0] := 3)", "<test>", "exec")

    def test_tuple_target_invalid(self):
        with self.assertRaises(SyntaxError):
            compile("((a, b) := (1, 2))", "<test>", "exec")

    def test_comprehension_iterable_restriction(self):
        with self.assertRaises(SyntaxError):
            compile("[x for x in (z := [1, 2])]", "<test>", "exec")

    def test_comprehension_iteration_collision(self):
        with self.assertRaises(SyntaxError):
            compile("[(x := x + 1) for x in range(2)]", "<test>", "exec")

    def test_class_comprehension_restriction(self):
        with self.assertRaises(SyntaxError):
            compile("class C:\n    x = [(z := y) for y in range(3)]", "<test>", "exec")

    def test_comprehension_outer_scope(self):
        def run():
            latest = None
            values = [(latest := x) for x in [3, 4]]
            return values, latest, "x" in locals()
        self.assertEqual(run(), ([3, 4], 4, False))

    def test_comprehension_empty_does_not_replace_binding(self):
        def run():
            witness = "initial"
            output = [(witness := x) for x in []]
            return output, witness
        self.assertEqual(run(), ([], "initial"))

    def test_generator_laziness(self):
        def run():
            witness = None
            generator = ((witness := x) for x in [1, 2])
            initial = witness
            next(generator)
            return initial, witness
        self.assertEqual(run(), (None, 1))

    def test_any_short_circuits(self):
        calls = []
        def inspect(value):
            calls.append(value)
            return value
        self.assertTrue(any((w := inspect(x)) > 2 for x in [1, 3, 9]))
        self.assertEqual(w, 3)
        self.assertEqual(calls, [1, 3])

    def test_and_skips_second_branch(self):
        calls = []
        def side_effect():
            calls.append(1)
            return True
        result = False and (assigned := side_effect())
        self.assertFalse(result)
        self.assertEqual(calls, [])

    def test_lambda_local_binding(self):
        fn = lambda x: (inner := x * 2)
        self.assertEqual(fn(3), 6)
        self.assertNotIn("inner", globals())

    def test_dict_key_before_value(self):
        trace = []
        def key():
            trace.append("key")
            return "K"
        def value(k):
            trace.append("value")
            return k * 2
        d = {(captured := key()): value(captured) for _ in range(1)}
        self.assertEqual(d, {"K": "KK"})
        self.assertEqual(trace, ["key", "value"])

    def test_ast_named_expression(self):
        node = ast.parse("(x := 4)", mode="eval").body
        self.assertIsInstance(node, ast.NamedExpr)
        self.assertIsInstance(node.target.ctx, ast.Store)

    def test_failed_rhs_preserves_previous_binding(self):
        def run():
            x = "before"
            def fail():
                raise ValueError
            try:
                (x := fail())
            except ValueError:
                pass
            return x
        self.assertEqual(run(), "before")

    def test_single_evaluation_of_capture(self):
        calls = []
        def f():
            calls.append(1)
            return 2
        if (result := f()) == 2:
            self.assertEqual(result, 2)
        self.assertEqual(len(calls), 1)


class QuoteGateTests(unittest.TestCase):
    def setUp(self):
        self.policy = POLICY

    def test_reference_event_count(self):
        self.assertEqual(len(stream_report.decisions), 14)

    def test_reference_accepted_count(self):
        self.assertEqual(stream_report.accepted_count, 6)

    def test_reference_rejected_count(self):
        self.assertEqual(stream_report.rejected_count, 8)

    def test_reason_distribution(self):
        self.assertEqual(Counter(x.reason for x in stream_report.decisions), reasons)

    def test_last_aapl_mid(self):
        last = dict(stream_report.last_accepted)
        self.assertEqual(last["AAPL"].mid, Decimal("100.15"))

    def test_last_msft_mid(self):
        last = dict(stream_report.last_accepted)
        self.assertEqual(last["MSFT"].mid, Decimal("202.05"))

    def test_zero_spread_is_accepted(self):
        r = assess_quote_stream([quote_record("Q0")], self.policy)
        self.assertEqual(r.accepted_count, 1)
        self.assertEqual(r.decisions[0].spread_bps, Decimal("0"))

    def test_empty_mapping_counted(self):
        r = assess_quote_stream([{}, quote_record("Q1")], self.policy)
        self.assertEqual((len(r.decisions), r.accepted_count), (2, 1))

    def test_missing_ask_rejected(self):
        item = quote_record("Q1")
        del item["ask"]
        self.assertEqual(assess_quote_stream([item], self.policy).rejected_count, 1)

    def test_bool_sequence_rejected(self):
        self.assertIsNone(parse_market_quote(quote_record("X", sequence=True)))

    def test_bool_bid_rejected(self):
        self.assertIsNone(parse_market_quote(quote_record("X", bid=True)))

    def test_nonfinite_bid_rejected(self):
        self.assertIsNone(parse_market_quote(quote_record("X", bid="NaN")))

    def test_infinite_ask_rejected(self):
        self.assertIsNone(parse_market_quote(quote_record("X", ask="Infinity")))

    def test_zero_price_rejected(self):
        self.assertIsNone(parse_market_quote(quote_record("X", bid="0")))

    def test_crossed_book_rejected(self):
        r = assess_quote_stream([quote_record("X", bid="101", ask="100")], self.policy)
        self.assertEqual(r.decisions[0].reason, "CROSSED_BOOK")

    def test_duplicate_id_rejected(self):
        r = assess_quote_stream([
            quote_record("X", sequence=1),
            quote_record("X", sequence=2),
        ], self.policy)
        self.assertEqual(r.decisions[1].reason, "DUPLICATE_ID")

    def test_stale_sequence_rejected(self):
        r = assess_quote_stream([
            quote_record("X", sequence=2),
            quote_record("Y", sequence=1),
        ], self.policy)
        self.assertEqual(r.decisions[1].reason, "STALE_SEQUENCE")

    def test_wide_spread_rejected(self):
        r = assess_quote_stream([quote_record("X", bid="99", ask="101")], self.policy)
        self.assertEqual(r.decisions[0].reason, "WIDE_SPREAD")

    def test_mid_jump_rejected(self):
        r = assess_quote_stream([
            quote_record("X", sequence=1, bid="100", ask="100"),
            quote_record("Y", sequence=2, bid="110", ask="110"),
        ], self.policy)
        self.assertEqual(r.decisions[1].reason, "MID_JUMP")

    def test_rejected_outlier_does_not_poison_reference(self):
        r = assess_quote_stream([
            quote_record("X", sequence=1, bid="100", ask="100"),
            quote_record("Y", sequence=2, bid="200", ask="200"),
            quote_record("Z", sequence=3, bid="101", ask="101"),
        ], self.policy)
        self.assertEqual([x.status for x in r.decisions], ["ACCEPT", "REJECT", "ACCEPT"])
        self.assertEqual(dict(r.last_accepted)["AAPL"].sequence, 3)

    def test_per_ticker_independence(self):
        r = assess_quote_stream([
            quote_record("X", ticker="AAPL", sequence=5),
            quote_record("Y", ticker="MSFT", sequence=1),
        ], self.policy)
        self.assertEqual(r.accepted_count, 2)

    def test_empty_stream(self):
        r = assess_quote_stream(iter(()), self.policy)
        self.assertEqual((len(r.decisions), r.accepted_count), (0, 0))

    def test_deterministic_replay(self):
        self.assertEqual(
            assess_quote_stream(iter(EXAMPLE_EVENTS), self.policy),
            assess_quote_stream(iter(EXAMPLE_EVENTS), self.policy),
        )

    def test_input_preservation(self):
        original = [dict(row) for row in EXAMPLE_EVENTS]
        assess_quote_stream(EXAMPLE_EVENTS, self.policy)
        self.assertEqual(EXAMPLE_EVENTS, original)

    def test_negative_policy_rejected(self):
        with self.assertRaises(ValueError):
            QuoteGatePolicy(Decimal("-1"), Decimal("150"))

    def test_infinite_policy_rejected(self):
        with self.assertRaises(ValueError):
            QuoteGatePolicy(Decimal("Infinity"), Decimal("150"))

    def test_exact_spread_threshold_allowed(self):
        bid = Decimal("99.9")
        ask = Decimal("100.1")
        # Spread is exactly 20 bps (0.2 / 100 * 10000).
        r = assess_quote_stream([
            quote_record("X", bid=bid, ask=ask)
        ], self.policy)
        self.assertEqual(r.accepted_count, 1)
        self.assertEqual(r.decisions[0].spread_bps, Decimal("20"))

    def test_exact_jump_threshold_allowed(self):
        r = assess_quote_stream([
            quote_record("X", sequence=1, bid="100", ask="100"),
            quote_record("Y", sequence=2, bid="101.5", ask="101.5"),
        ], self.policy)
        self.assertEqual(r.accepted_count, 2)
        self.assertEqual(r.decisions[1].jump_bps, Decimal("150"))

    def test_strictly_increasing_sequences(self):
        grouped: dict[str, list[int]] = {}
        for item in stream_report.accepted_quotes:
            grouped.setdefault(item.ticker, []).append(item.sequence)
        for values in grouped.values():
            self.assertTrue(all(a < b for a, b in zip(values, values[1:])))

    def test_all_accepted_spreads_within_limit(self):
        for decision in stream_report.decisions:
            if decision.status == "ACCEPT":
                self.assertIsNotNone(decision.spread_bps)
                self.assertLessEqual(decision.spread_bps, self.policy.max_spread_bps)

    def test_all_accepted_jumps_within_limit(self):
        for decision in stream_report.decisions:
            if decision.status == "ACCEPT" and decision.jump_bps is not None:
                self.assertLessEqual(decision.jump_bps, self.policy.max_jump_bps)

    def test_acceptance_conservation(self):
        self.assertEqual(
            stream_report.accepted_count + stream_report.rejected_count,
            len(stream_report.decisions),
        )

    def test_valid_identifiers_unique_among_accepted(self):
        ids = [x.event_id for x in stream_report.accepted_quotes]
        self.assertEqual(len(ids), len(set(ids)))


all_tests = unittest.TestSuite([
    unittest.defaultTestLoader.loadTestsFromTestCase(WalrusSemanticsTests),
    unittest.defaultTestLoader.loadTestsFromTestCase(QuoteGateTests),
])
test_result = unittest.TextTestRunner(verbosity=1).run(all_tests)
assert test_result.wasSuccessful()
print(f"PASS: {test_result.testsRun} unit tests, 0 failed, 0 errors")

.

.

.

.

.

.

... (47 outputs left out)

.

.

----------------------------------------------------------------------
Ran 55 tests in 0.017s

OK


PASS: 55 unit tests, 0 failed, 0 errors


### **Analytical Interpretation**

**1. Grammar vs. application behavior.** The language tests verify scoping, evaluation order, grammar restrictions, and short-circuiting; the market-data tests separately verify numerical and policy semantics. **2. Explicit zeros.** The zero-spread test ensures that valid numerical zero is never mistaken for missing data. **3. Temporal invariant.** Accepted sequence numbers increase strictly per asset; rejected observations do not alter the accepted baseline. **4. Threshold boundaries.** Both limits are implemented with strict `>` comparisons, making the exact threshold permissible under the stated policy. **5. Error-path integrity.** Missing data, invalid types, and nonfinite inputs produce controlled rejections. **6. Reference implementation limits.** Tests establish correctness for the covered cases, not complete real-world exchange-data validity, production security, or statistically significant trading outcomes.

## 2.7.16 Final Analytical Interpretation

### 1. Assignment expression semantics

The expression `NAME := expr` evaluates `expr`, binds the resulting object to a **simple name**, and yields that object. This is not a generalized assignment target and does not replace all ordinary assignment statements.

### 2. Precedence and explicit grouping

Named expressions have very low precedence and require parentheses in specific syntactic contexts. Parentheses are often valuable even where optional, because they distinguish **what is being bound** from **what is subsequently compared**.

### 3. Scope, comprehensions, and nonlocal effects

A walrus inside a comprehension can bind a name in the **containing scope**, whereas the comprehension's iteration variable remains isolated by language scoping rules. This can produce observable state outside the comprehension, including the last evaluated rather than last selected value.

### 4. Laziness and conditional execution

A named expression may never execute because an iterable is empty, a generator is not advanced, an `and` / `or` branch short-circuits, or the RHS raises. Code must not assume a fresh binding occurred.

### 5. Value truthiness vs. data validity

A valid return, spread, count, or tracking error may be `0`. The guard `is not None` separates a legitimate zero from missing data. Truthiness-based filters can systematically bias the retained sample.

### 6. Quantitative streaming implications

The quality-gate example uses named expressions for **single-evaluation capture** and explicit guards. Its accepted events obey stated per-ticker sequence, spread, and jump constraints. Rejected events do not mutate the last-accepted reference. Every input event receives a decision.

### 7. Numerical precision

`Decimal` avoids unintended binary floating-point conversion during the example's spread calculations, but the precision context, source encoding, rounding policy, and market conventions remain part of the financial specification.

### 8. Engineering trade-offs

The walrus is valuable when it makes data flow shorter **and clearer**. Ordinary statements remain preferable for complicated validation, multi-branch state changes, or opaque side effects. Code review should examine variable lifetime, conditional binding, exception paths, and unintended scope leakage.

### 9. Research limitations

A syntactically and numerically correct quality gate does not validate market-data provenance or imply that a trading signal is profitable. Production designs need exchange-specific sequencing, clock definitions, stale-data policies, market-session controls, persistence, and operational monitoring.

### Final assessment

**Expert understanding of `:=` means knowing exactly when a name is bound, where the name lives, whether evaluation occurs, and which side effects and numerical values can follow from that choice.** Readability and correctness remain the final design criteria.

## Advanced Practice Exercises — Expert Level

These exercises deliberately omit complete solutions. For each exercise, supply runnable code, deterministic assertions, at least one failure case, and a separate `### **Analytical Interpretation**` discussing the language semantics and financial implications.

### Exercise 1 — Assignment Expression Grammar Auditor

1. Construct a corpus of at least 30 syntax examples with valid and invalid named-expression placements.
2. Include lambda, conditional-expression, comprehension-filter, keyword-argument, `assert`, `with`, `if`, `while`, and ordinary assignment contexts.
3. Use `compile()` and `ast.parse()` only; **do not execute arbitrary source strings**.
4. Report line/column locations where available.
5. Avoid asserting exact `SyntaxError` text across Python versions.

### Exercise 2 — Comprehension Scope and Lifetime Study

1. Compare list, set, dictionary, and generator comprehensions.
2. Prove the surrounding-function binding behavior of `:=`.
3. Distinguish last evaluated input from last accepted output.
4. Include empty comprehensions and aborted generators.
5. Explain leakage hazards in long-lived notebooks.

### Exercise 3 — Data-Quality Zero Preservation

1. Build a validator for returns, spread, volume, and risk score.
2. Treat zero as valid, missing `None` as invalid, and NaN as invalid.
3. Compare correct `is not None` checks with erroneous truthiness guards.
4. Quantify how incorrect filters change sample size and descriptive statistics.

### Exercise 4 — Lazy Witness and Counterexample Capture

1. Build `any()` and `all()` checks with named-expression witnesses.
2. Track exactly how many observations were evaluated.
3. Verify early exits, empty input, and repeated values.
4. Explain why a witness is **not** a full set of all violations.

### Exercise 5 — Key/Value Evaluation and Mapping Collision Policy

1. Use `:=` to normalize dictionary-comprehension keys.
2. Trace key-before-value evaluation.
3. Construct duplicate normalized tickers.
4. Implement explicit **reject**, **overwrite**, and **aggregate** policies in separate functions.
5. Verify deterministic and financially appropriate behavior.

### Exercise 6 — Side-Effect and Exception Robustness

1. Build a metric producer with a call counter and controlled failures.
2. Verify one calculation per evaluated record.
3. Investigate what the capture refers to after an exception.
4. Identify stale bindings across repeated notebook executions.
5. Compare a walrus implementation with explicit statements.

### Exercise 7 — Streaming Sentinel and Record Completeness

1. Stream events that include `0`, `{}`, `False`, empty strings, and `None`.
2. Use a unique identity sentinel.
3. Prove that no valid event is dropped.
4. Define a clear contract for `None` as a data value versus end-of-stream.
5. Analyze how the choice of sentinel changes correctness.

### Exercise 8 — Static Walrus Binding Inspector

1. Build an AST visitor that records all `NamedExpr` nodes.
2. Report the containing function and comprehension context of each capture.
3. Use `symtable` to classify captured names.
4. Flag captures that may never execute.
5. Do **not** claim the tool proves runtime initialization or complete safety.

### Exercise 9 — Point-in-Time Quote State Integrity

1. Extend the quality gate with per-symbol session identifiers.
2. Reject stale sessions and out-of-order sequence numbers.
3. Keep last accepted mid separate from latest observed mid.
4. Introduce a rejection report with financial-unit annotations.
5. Verify replay determinism and input preservation.

### Exercise 10 — Numerical Sensitivity and Precision

1. Re-implement spread and jump calculations using both float and `Decimal`.
2. Compare near-boundary quote decisions.
3. Investigate precision-context effects and rounding policy.
4. Define an auditable and explicit boundary comparison rule.
5. Document why numerical type choice affects financial outcomes.

### Exercise 11 — Walrus Refactoring and Readability Review

1. Select five overcomplicated named-expression examples.
2. Rewrite them with explicit assignment statements.
3. Prove behavioral equivalence with tests.
4. Compare measured function-call counts.
5. Explain which version is most maintainable and why.

### Exercise 12 — Property-Based Invariants Without External Libraries

1. Generate deterministic synthetic event streams using a seeded pseudorandom generator.
2. Assert accepted/rejected conservation and strict accepted sequence ordering.
3. Include duplicate IDs and deliberately malformed records.
4. Prove accepted price spreads and jumps respect policy limits.
5. Shrink failing test inputs manually and report the smallest counterexample.

## Advanced Challenge — Auditable Streaming Quantitative Research Ingestion Framework

### Objective

Build a modular, testable, and auditable Python application that transforms a market-data stream into normalized, quality-checked research observations while making the timing and scope of every assignment explicit.

### Required architecture

**1. Ingestion and normalization:** Iterators/generators, explicit sentinel handling, finite numeric inputs, required-field validation, typed records.

**2. Assignment-expression discipline:** Use `:=` only for meaningful single evaluations; document all capture scopes and potential short-circuit paths.

**3. Temporal integrity:** Session boundaries, exchange sequence numbers, per-symbol accepted reference, rejection of stale events, deterministic replay.

**4. Quality measures:** Crossed/locked market policy, spread in bps, relative mid jump, threshold boundary tests, numerical precision strategy.

**5. Operational observability:** Structured rejection reasons, event IDs, stable counts, versioned policy snapshots, deterministic run summaries.

**6. Static code audit:** `ast.NamedExpr` inspection, suspicious comprehension captures, optional uninitialized targets, disallowed syntax placement diagnostics.

**7. Test and verification:** Unit tests, adversarial malformed records, zero-value preservation, empty-stream behavior, single evaluation, side-effect ordering, temporal invariants, replay consistency.

**8. Extension design:** Explicit interfaces for future timestamp provenance, real market-data feed integration, concurrent ingestion, persistent storage, and downstream feature generation.

### Expected deliverables

1. A packaged Python module for quote parsing and validation.
2. A streaming-quality evaluation engine.
3. Immutable data contracts and a versioned configuration format.
4. A diagnostic audit report with bps metrics and reason codes.
5. Deterministic reference data and expected results.
6. A static analysis utility for assignment-expression usage.
7. A comprehensive test suite and benchmark notes.
8. A technical document separating **syntax correctness, numerical validity, temporal consistency, and market realism**.

### Evaluation rubric

| Dimension | Weight |
|---|---:|
| Named-expression semantic correctness | 25% |
| Temporal and state integrity | 20% |
| Financial numerical accuracy | 20% |
| Automated testing and failure coverage | 15% |
| Architecture and maintainability | 10% |
| Auditability and documentation | 10% |

### Completion criterion

The system should preserve every input event's disposition, never confuse zero with missing data, produce identical results on replay, reject stale or corrupted records without changing accepted reference state, and document every significant use of assignment expressions.

## Appendix — Reference Documentation and Next Steps

### Primary references

- [PEP 572 — Assignment Expressions](https://peps.python.org/pep-0572/)
- [Python Language Reference — Expressions](https://docs.python.org/3/reference/expressions.html)
- [Python Standard Library — `ast`](https://docs.python.org/3/library/ast.html)
- [Python Standard Library — `symtable`](https://docs.python.org/3/library/symtable.html)
- [PEP 8 — Style Guide for Python Code](https://peps.python.org/pep-0008/)

### Conceptual progression

- **2.6 Annotated Assignment and Type Hints:** Declaring contracts without automatic runtime enforcement.
- **2.7 Assignment Expressions and the Walrus Operator:** Explicit name capture within expressions, without introducing a new scope.
- **2.8 Namespaces, Scope, and Binding Rules:** Comprehensive treatment of LEGB, closures, `global`, `nonlocal`, scope restrictions, and modern annotation scopes.
- **2.9 Object Identity, Aliasing, and Mutability:** Reference sharing and mutation consequences.

**Core lesson:** Use the walrus operator only when it improves clarity while preserving correct value handling, scope behavior, and side-effect semantics.